In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers torch scikit-learn nltk

In [ ]:
import torch
import numpy as np
import pandas as pd
import re
import nltk

from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

nltk.download("punkt")
nltk.download("punkt_tab")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
file_path = "/content/drive/MyDrive/preprocessed_casesumm.csv"
df = pd.read_csv(file_path)

print("Columns:", df.columns.tolist())
print("Total documents:", len(df))

Columns: ['Citation', 'Syllabus', 'Original_Opinion', 'Clean_Opinion']
Total documents: 2000


In [ ]:
def get_sentence_embeddings_batch(sentences, batch_size=16):
    all_embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # CLS token embedding
        emb = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(emb.cpu())

    return torch.cat(all_embeddings).numpy()

In [ ]:
def textrank_scores(sim_matrix, eps=1e-6, max_iter=100):
    n = sim_matrix.shape[0]
    scores = np.ones(n) / n

    for _ in range(max_iter):
        new_scores = np.ones(n)
        for i in range(n):
            for j in range(n):
                if sim_matrix[j, i] > 0:
                    new_scores[i] += scores[j] * sim_matrix[j, i] / (sim_matrix[j].sum() + eps)
        scores = new_scores / new_scores.sum()

    return scores

In [ ]:
def is_boilerplate(sentence):
    patterns = [
        r'case\s+\d', r'docket', r'pacer', r'civil cover sheet',
        r'notice of electronic filing', r'u\.s\. district court'
    ]
    s = sentence.lower()
    return any(re.search(p, s) for p in patterns)

def is_legal_noise(sentence):
    s = sentence.lower()

    if len(s.split()) < 10:
        return True

    if sentence.isupper() and len(sentence.split()) < 20:
        return True

    if re.search(r'\b(plaintiff|defendant|petitioner|respondent)\b', s) and len(s.split()) < 15:
        return True

    return False

In [ ]:
def summarize_document_fast(doc, ratio=0.15, lambda_param=0.65):

    sentences = [
        s.strip()
        for s in sent_tokenize(doc)
        if 10 <= len(s.split()) <= 40
        and not is_boilerplate(s)
        and not is_legal_noise(s)
    ]

    if len(sentences) < 2:
        return doc

    # ---- Clean for embeddings only ----
    def clean(s):
        s = re.sub(r'^\d+\s+', '', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s if len(s.split()) >= 12 else ""

    clean_sentences = [clean(s) for s in sentences]
    clean_sentences = [s for s in clean_sentences if s]

    if len(clean_sentences) < 2:
        return " ".join(sentences[:3])

    embeddings = get_sentence_embeddings_batch(clean_sentences)

    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, 0)

    scores = textrank_scores(sim_matrix)

    position_weight = 0.15  # small boost

    for i in range(len(scores)):
        scores[i] += position_weight * (1 - i / len(scores))

    k = min(6, len(clean_sentences))
    labels = KMeans(n_clusters=k, random_state=42, n_init=5).fit_predict(embeddings)

    selected = []
    for c in range(k):
        idx = np.where(labels == c)[0]

        if len(idx) == 0:
            continue   # skip empty clusters safely

        best = idx[np.argmax(scores[idx])]
        selected.append(best)

    tail_start = int(len(clean_sentences) * 0.8)
    tail_candidates = list(range(tail_start, len(clean_sentences)))

    if tail_candidates:
        best_tail = max(tail_candidates, key=lambda i: scores[i])
        if best_tail not in selected:
            selected.append(best_tail)

    word_limit = int(len(doc.split()) * ratio)
    current_words = sum(len(clean_sentences[i].split()) for i in selected)

    remaining = list(set(range(len(clean_sentences))) - set(selected))

    while remaining and current_words < word_limit:
        mmr_vals = []
        for i in remaining:
            sim_to_sel = np.mean([sim_matrix[i][j] for j in selected])
            position_bonus = 0.15 * (1 - i / len(clean_sentences))
            mmr = lambda_param * scores[i] + position_bonus - (1 - lambda_param) * sim_to_sel
            mmr_vals.append((mmr, i))

        _, best = max(mmr_vals)
        sent_len = len(clean_sentences[best].split())

        if current_words + sent_len <= word_limit:
            selected.append(best)
            current_words += sent_len

        remaining.remove(best)

    selected.sort()
    return " ".join(clean_sentences[i] for i in selected)

In [ ]:
doc = df["Original_Opinion"].iloc[0]
summary = summarize_document_fast(doc)

print("Original words:", len(doc.split()))
print("Summary words:", len(summary.split()))
print("\nSUMMARY:\n", summary)

Original words: 447
Summary words: 161

SUMMARY:
 This motion to dismiss is made because, as is alleged, (1) the decree appealed from is not a final decree; and (2) the transcript is not properly certified. The case is in some particulars different from that of the St. Louis, I. M. & S. Ry. Co. v. Southern Express Co., just decided, but in our opinion the differences do not materially affect the present question. Still, in this, as in that, the reference is in respect to matters affecting the administration of the cause, and does not involve the merits. As the decree stands, the express company can require the railway company to carry at the rate which has been fixed. The clerk certifies the transcript sent up to be 'a true, full, and perfect copy from the record of all the proceedings in the suit.' If, in point of fact, the certificate is not true, the remedy is by certiorari to supply deficiencies, and not by motion to dismiss.


In [ ]:
import time

start = time.time()
summary = summarize_document_fast(doc, ratio=0.15)
print("Time taken:", round(time.time() - start, 2), "seconds")

Time taken: 0.13 seconds


In [ ]:
summaries = []

total_docs = len(df)
print("Total documents to process:", total_docs)

for i in range(10):
    doc = df["Original_Opinion"].iloc[i]
    summary = summarize_document_fast(doc, ratio=0.15)
    summaries.append(summary)

# Add summaries as a new column
df.loc[:9, "summary"] = summaries

Total documents to process: 2000


In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/summarized_casesumm.csv"
df.to_csv(OUTPUT_PATH, index=False)

print("Saved summarized dataset to:", OUTPUT_PATH)

Saved summarized dataset to: /content/drive/MyDrive/summarized_casesumm.csv
